In [ ]:
import os
import librosa
import librosa.display
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
DATA_PATH = 'free-spoken-digit-dataset/recordings/'
print(f"Dataset loaded. Total files: {len(os.listdir(DATA_PATH))}")

In [ ]:
file_path = os.path.join(DATA_PATH, '0_jackson_0.wav')
signal, sr = librosa.load(file_path, sr=None)

# display audio
print("Playing: Digit 0 by Jackson")
ipd.display(ipd.Audio(signal, rate=sr))

# waveform plot
plt.figure(figsize=(10, 4))
librosa.display.waveshow(signal, sr=sr, alpha=0.8)
plt.title('Waveform: Digit 0')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.show()

In [ ]:
# short time fourier transform
stft = librosa.stft(signal)
stft_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

# mel spectrogram
mel_spec = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 10))

# Linear Spectrogram
img1 = librosa.display.specshow(stft_db, x_axis='time', y_axis='hz', ax=ax[0])
ax[0].set_title('Linear-frequency Spectrogram (dB)')
fig.colorbar(img1, ax=ax[0], format="%+2.f dB")

# Mel Spectrogram
img2 = librosa.display.specshow(mel_spec_db, x_axis='time', y_axis='mel', ax=ax[1])
ax[1].set_title('Mel-frequency Spectrogram (dB)')
fig.colorbar(img2, ax=ax[1], format="%+2.f dB")

plt.tight_layout()
plt.show()

In [ ]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)
    # Extract MFCCs (standard is 13-20)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    # We take the mean across time frames to get a fixed-size vector
    return np.mean(mfccs.T, axis=0)

# Example: Compare digit '0' and digit '7'
feat_0 = extract_features(os.path.join(DATA_PATH, '0_jackson_0.wav'))
feat_7 = extract_features(os.path.join(DATA_PATH, '7_jackson_0.wav'))

plt.figure(figsize=(8, 4))
plt.plot(feat_0, label='Digit 0', marker='o')
plt.plot(feat_7, label='Digit 7', marker='x')
plt.title('MFCC Feature Comparison')
plt.legend()
plt.show()

In [ ]:
def prepare_dataset(data_path):
    features = []
    labels = []

    for file_name in os.listdir(data_path):
        if file_name.endswith('.wav'):
            # The label is the first character of the filename (e.g., '0' from '0_jackson_0.wav')
            label = int(file_name[0])
            file_path = os.path.join(data_path, file_name)

            # Extract MFCC features
            mfccs = extract_features(file_path)

            features.append(mfccs)
            labels.append(label)

    return np.array(features), np.array(labels)

print("Extracting features from all files... (this may take a minute)")
X, y = prepare_dataset(DATA_PATH)

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling features is critical for SVM performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the SVM Classifier
clf = SVC(kernel='rbf', C=10, gamma='scale')
clf.fit(X_train_scaled, y_train)

print("Model training complete!")

In [ ]:
# Predict on the test set
y_pred = clf.predict(X_test_scaled)

# Print Accuracy and F1-Score
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix: Digit Classification')
plt.show()

# Exercise 7: Audio Analysis Report

### Dataset Overview
**Dataset:** Free Spoken Digit Dataset (FSDD)

For this analysis, I selected the **Free Spoken Digit Dataset (FSDD)**. This dataset consists of 3,000 recordings of spoken digits (0-9) generated by multiple speakers. I chose FSDD because it offers an excellent environment for testing speaker invariance—the ability to recognize the same word regardless of the speaker's voice pitch or accent—which is a fundamental challenge in audio classification.

---

### Analysis & Visualizations
Using `librosa` and `matplotlib`, I processed the raw audio to visualize the signal characteristics:

* **Waveform Analysis:** I plotted the amplitude over time to visualize the raw signal structure. This provided a baseline view of the audio density and silence intervals.
* **Spectrograms (dB Scale):** I converted the signals to the frequency domain to observe spectral content. Crucially, I applied the **Decibel (dB) scale** (`amplitude_to_db`). Without this logarithmic scaling, low-energy frequencies were invisible; the dB scale aligns the visualization with human auditory perception, revealing the full spectral texture.
* **Mel-Spectrograms:** I utilized the Mel scale to map the frequencies, emphasizing the lower frequencies which are more critical for human speech intelligibility and recognition tasks.

---

### Classification Process
To classify the digits, I implemented a pipeline focusing on feature efficiency:

1.  **Feature Extraction (MFCCs):** Instead of using raw audio, I extracted **Mel-frequency cepstral coefficients (MFCCs)**. This creates a compact numerical "fingerprint" of the audio, capturing the spectral envelope relevant to speech.
2.  **Preprocessing:** I applied `StandardScaler` to normalize the features. This was a critical step that significantly improved the convergence and accuracy of the model.
3.  **Model:** The features were fed into a **Support Vector Machine (SVM)** classifier.

---

### Observations & Insights
* **Visual Patterns:** There is a distinct spectral difference between digits. "Sharp" digits like **"six"** or **"seven"** exhibited high-frequency spikes in the spectrogram, whereas "soft" digits like **"zero"** were concentrated in lower frequency bands.
* **Model Performance:** The SVM proved highly effective for this task. The confusion matrix indicated that most errors occurred between phonetically similar digits, while distinctive sounds were classified with high precision.

---

### Challenges & Solutions

**Challenge 1: Variable Audio Lengths**
* **Problem:** The raw recordings varied in duration, making it impossible to feed them directly into the SVM, which requires a fixed input vector size.
* **Solution:** I calculated the **mean of the MFCCs across the time axis**. This collapsed the temporal dimension, resulting in a fixed-size feature vector for every recording regardless of its original length.

**Challenge 2: Visibility in Spectrograms**
* **Problem:** The raw power spectrograms were too dark, masking key details of the frequency distribution.
* **Solution:** As mentioned in the visualization section, I utilized `librosa.amplitude_to_db` to convert the magnitude to the decibel scale. This dynamic range compression illuminated hidden details in the speech signal.